## What is a candidate panel, and why does it exist?

A **candidate panel** is the precomputed, persisted output of the expensive part of the pipeline — causal residualization + pair candidate construction — decoupled from simulation.

**Rationale:** residual computation (rolling/expanding regressions per stock) and candidate discovery (hedge ratios, cointegration checks, half-life/kappa filters) are the costly steps. Simulation — sizing, entry/exit logic, sweeping parameters like `entry_z` — is comparatively cheap and gets run *many times over* against the same candidates during research. A candidate panel is computed once per (sector, residual config) and reused across every downstream simulation that needs it.

**What it consists of:**
- **Panel** (`.panel.parquet`) — one row per `(pair, asof_date)`, with hedge ratios, spread diagnostics, and filter-pass metadata for every candidate pair discovered at that date.
- **`residual_params`** (`_residual_params.pkl`) — the fitted residualization parameters needed to reconstruct residual series without refitting.
- **`meta.json`** — the residual config used to produce the panel, plus date range and row counts — lets downstream code confirm it's loading the right panel for a given `residual_key`.

Stock/spread-level **`series/`** is a separate, optional persistence step — not part of the panel itself, and recomputed on demand if absent.

## How to create a candidate panel

This notebook builds one panel for the **Materials** sector, persists it, and loads it back.

The entry point is `run_panel_batch(PanelBatchConfig(...))` from `src.candidates.panel_batch` — the same function the `residuals` stage of `run_me.py` calls.

## Panel creation has three independent steps

1. **Fit residuals** — causal regression of each stock against market/sector, per `CausalResidualConfig.mode`.
2. **Fit hedge ratios** — OLS or PCA on the fitted residuals, over a window of `hedge_ratio_lb` days. Unweighted.
3. **Compute MR diagnostics** — kappa, half-life, mr_score, over a window of `mr_diag_lb` days. Unweighted.

Steps 2 and 3 are independent of each other and of the residual-fitting axis — you can fit residuals on a long, stable window while scoring mean-reversion diagnostics on a shorter, more recent one, or vice versa.

## Residual modes — pick one

`CausalResidualConfig` is mode-discriminated: only the fields for the chosen `mode` may be set, everything else must stay `None` (the constructor raises otherwise).

| `mode` | weighting | window | mode-specific fields | `min_history` |
| --- | --- | --- | --- | --- |
| `EQ_ROLLING` | equal | rolling | `lb` | = `lb` |
| `EQ_EXPANDING` | equal | expanding | `min_lb_eq_exp` | = `min_lb_eq_exp` |
| `DECAY_EXPANDING` | exponential decay | expanding | `hl`, `min_lb_type_dec_exp`, `min_lb_dec_exp` | `hl * min_lb_dec_exp` (multiplier) or `min_lb_dec_exp` directly (absolute) |

`decay_rolling` doesn't exist — time-decay + rolling window is redundant with `DECAY_EXPANDING` and deliberately excluded.

`subtract_risk_free` has no default — state it explicitly for every config; it's material to the fit and to the resulting `residual_key`.

## Knobs

`PanelBatchConfig.residual_configs` takes a **list** of already-built `CausalResidualConfig` objects — the sweep *is* that list. Here we build exactly one, for the mode selected below.

In [1]:
from src.candidates.panel_batch import PanelBatchConfig, run_panel_batch
from src.residuals.causal_residuals import CausalResidualConfig, ResidualMode, AbsOrMult

# ── KNOB: pick the residual mode ─────────────────────────────────────────
RESIDUAL_MODE = ResidualMode.DECAY_EXPANDING   # try: EQ_ROLLING, EQ_EXPANDING

# ── KNOB: DECAY_EXPANDING params (ignored unless RESIDUAL_MODE == DECAY_EXPANDING) ──
RESIDUAL_HL = 504                              # -> exp_hl504
MIN_HIST_TYPE_DEC_EXP = AbsOrMult.MULTIPLIER   # MULTIPLIER or ABSOLUTE
MIN_HIST_VALUE_DEC_EXP = 2                     # multiplier: min_history = hl*2 = 1008 -> mh1008
                                                # absolute:   min_history = this value directly

# ── KNOB: EQ_ROLLING params (ignored unless RESIDUAL_MODE == EQ_ROLLING) ──
RESIDUAL_LB = 126                              # -> rol_lb126; min_history = lb, always

# ── KNOB: EQ_EXPANDING params (ignored unless RESIDUAL_MODE == EQ_EXPANDING) ──
MIN_HIST_EQ_EXP = 252                          # -> exp_mh252; no swept axis in this mode

# ── KNOB: shared across all modes — steps 2 and 3, independent of each other ──
HEDGE_RATIO_LB = 252                           # window for OLS/PCA hedge-ratio fit
MR_DIAG_LB = 252                               # window for mean-reversion diagnostics
                                                # try MR_DIAG_LB=63 or 126 with HEDGE_RATIO_LB=252
                                                # held fixed, to test shorter-window diagnostics

# ── KNOB: risk-free rate subtraction — no default, must be stated ────────
SUBTRACT_RISK_FREE = True                      # -> _rf suffix on residual_key

# ── KNOB: sector / universe; None=all available sectors ──────────────────
SELECTED_SECTORS = ["materials"]

# ── KNOB: notebook speed cap — omit / set None for full history ──────────
MAX_STEPS = 8

In [2]:
# Build the ONE CausalResidualConfig for the chosen mode — only the fields
# relevant to RESIDUAL_MODE are set; the rest must stay None or __post_init__ raises.
match RESIDUAL_MODE:
    case ResidualMode.DECAY_EXPANDING:
        residual_cfg = CausalResidualConfig(
            mode=RESIDUAL_MODE,
            subtract_risk_free=SUBTRACT_RISK_FREE,
            hl=RESIDUAL_HL,
            min_lb_type_dec_exp=MIN_HIST_TYPE_DEC_EXP,
            min_lb_dec_exp=MIN_HIST_VALUE_DEC_EXP,
        )
    case ResidualMode.EQ_ROLLING:
        residual_cfg = CausalResidualConfig(
            mode=RESIDUAL_MODE,
            subtract_risk_free=SUBTRACT_RISK_FREE,
            lb=RESIDUAL_LB,
        )
    case ResidualMode.EQ_EXPANDING:
        residual_cfg = CausalResidualConfig(
            mode=RESIDUAL_MODE,
            subtract_risk_free=SUBTRACT_RISK_FREE,
            min_lb_eq_exp=MIN_HIST_EQ_EXP,
        )

print(f"residual key: {residual_cfg.key}")

# residual_configs is a list — the sweep IS this list. One entry here.
cfg = PanelBatchConfig(
    residual_configs=[residual_cfg],
    hedge_ratio_lb=HEDGE_RATIO_LB,
    mr_diag_lb=MR_DIAG_LB,
    selected_sectors=SELECTED_SECTORS,
    frequency="W-FRI",
    max_steps=MAX_STEPS,               # notebook-only cap; omit for full history
    persist_result=True,
    persist_residual_params=True,
    persist_dir_template="howto",      # subdir under settings.CANDIDATE_PANELS_ROOT
)

# resolved_universe_dir()/resolved_data_path() fall back to settings when unset.
results = run_panel_batch(cfg)

residual key: exp_hl504_mh1008_rf
Batch 20260724_1204
Sectors: 1 [mat]
Residual configs: ['exp_hl504_mh1008_rf']
Total jobs: 1


Sectors:   0%|          | 0/1 [00:00<?, ?sector/s]

Loading universe materials_only_v1...


  mat / exp_hl504_mh1008_rf: 624/624 valid candidates

Done. 1 panels created.


`run_panel_batch` returns a dict keyed by `(group_id, residual_key)`. Each value is a `CandidatePanelResult` with a `.panel` DataFrame and a `.metadata` dict.

In [3]:
for key, result in results.items():
    print(f"key: {key}")
    print(f"rows: {len(result.panel)}  valid: {int(result.panel['is_valid'].sum())}")
    print(f"residual key: {result.metadata['residual_cfg']}")

next(iter(results.values())).panel.head()

key: ('materials', 'exp_hl504_mh1008_rf')
rows: 624  valid: 624
residual key: {'mode': <ResidualMode.DECAY_EXPANDING: 'decay_expanding'>, 'subtract_risk_free': True, 'remove_residual_pcs': 0, 'lb': None, 'hl': 504, 'min_lb_type_dec_exp': <AbsOrMult.MULTIPLIER: 'multiplier'>, 'min_lb_dec_exp': 2, 'min_lb_eq_exp': None, 'window_mode': 'expanding', 'min_history': 1008}


,candidate_type,candidate_subtype,weight_model,spread_id,group_id,n_legs,weights,adf_pvalue,mr_score,kappa,half_life,residual_std,spread_return_std,level_std,is_valid,candidate_id,asof_datetime,asof_date
0,pair,pca,pca,APD|AVY,materials,2,"{""APD"":1.0,""AVY"":-0.27601510977}",0.362243,1.455425,0.027276,25.411931,0.018741,0.018830,0.079880,True,d43c7577f9d2fb27,2009-08-14,2009-08-14
1,pair,pca,pca,APD|BALL,materials,2,"{""APD"":1.0,""BALL"":-8.605602202994}",0.427050,0.085537,0.011562,59.948202,0.135175,0.135585,0.994331,True,277764d34d72840b,2009-08-14,2009-08-14
2,pair,pca,pca,APD|CF,materials,2,"{""APD"":1.0,""CF"":-0.063262813943}",0.244270,2.312385,0.041635,16.648337,0.018005,0.018160,0.062296,True,4b5d98816095beed,2009-08-14,2009-08-14
3,pair,pca,pca,APD|ECL,materials,2,"{""APD"":-1.0,""ECL"":-9.238356540198}",0.009992,0.564488,0.088376,7.843182,0.156559,0.159895,0.384340,True,6364c222d3d9f733,2009-08-14,2009-08-14
4,pair,pca,pca,APD|IFF,materials,2,"{""APD"":1.0,""IFF"":-0.894969585636}",0.078408,2.335524,0.055329,12.527631,0.023690,0.023981,0.072448,True,1621dfbf08198b37,2009-08-14,2009-08-14


## Loading a persisted panel back

Persistence writes three artifacts per panel stem into the subdir: `{stem}.panel.parquet`, `{stem}.meta.json`, and `{stem}_residual_params.pkl`. `run_panel_batch` generates the timestamped stem internally, so recover it from the result metadata (the same trick `run_me.py::_extract_panel_stem` uses), then reload with `load_candidate_panel_result`.

In [4]:
from pathlib import Path
from src.candidates.candidate_panel import load_candidate_panel_result

(group_id, residual_key), result = next(iter(results.items()))

out_dir = Path(result.metadata["artifact_out_dir"])
stem = Path(result.metadata["residual_params_path"]).name.removesuffix("_residual_params.pkl")

reloaded = load_candidate_panel_result(out_dir=out_dir, stem=stem)
print(f"out_dir: {out_dir}")
print(f"stem:    {stem}")
print(f"reloaded panel: {reloaded.panel.shape}, valid: {int(reloaded.panel['is_valid'].sum())}")

out_dir: /home/nikolajnock/PycharmProjects/statarb_sim/artifacts/candidate_panels/howto
stem:    mat_pairs_pca_W-FRI_exp_hl504_20260724_1204
reloaded panel: (624, 18), valid: 624


## Out of scope: stock-residual & spread-level series

This notebook stops at the candidate panel (plus its fitted daily residual params). Precomputing and persisting the **stock-residual and spread-level series** — which the simulator later loads instead of recomputing residuals each step — is a **separate step** and is not covered here.

That step lives in `src.residuals.series.compute_and_persist_series`, and is driven end-to-end by `run_me.py` (`_persist_spread_series` for a single sector, `_persist_series_multi` for many) as part of its `residuals` stage, right after the panel is built.

See [`02_run_simulation.ipynb`](02_run_simulation.ipynb) for that step, and for running an actual simulation against a persisted panel.